<a href="https://colab.research.google.com/github/RajeevTricel/gemma-seo-agent/blob/main/notebooks/01_finetune_gemma_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Mon Apr 27 15:31:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/RajeevTricel/gemma-seo-agent.git
%cd gemma-seo-agent

fatal: destination path 'gemma-seo-agent' already exists and is not an empty directory.
/content/gemma-seo-agent


In [ ]:
!ls

agent  data  docs  notebooks  README.md  requirements.txt


In [ ]:
!ls data

eval.jsonl  train.jsonl


In [ ]:
!head -n 3 data/train.jsonl

{"messages":[{"role":"system","content":"You are a practical SEO and web performance assistant. Give clear priority-based fixes."},{"role":"user","content":"LCP is 5.2s and the hero image is 3.4MB. What should I fix first?"},{"role":"assistant","content":"Fix the hero image first because it is likely the main LCP element. Compress it, convert it to WebP or AVIF, preload it, set width and height, and do not lazy-load it if it appears above the fold."}]}
{"messages":[{"role":"system","content":"You are a practical SEO and web performance assistant. Give clear priority-based fixes."},{"role":"user","content":"PageSpeed shows render-blocking CSS and unused JavaScript. What should the developer do?"},{"role":"assistant","content":"Start with render-blocking CSS because it delays first paint. Inline critical CSS, defer non-critical styles, remove unused CSS, then reduce unused JavaScript by delaying third-party scripts and splitting large bundles."}]}
{"messages":[{"role":"system","content":

In [ ]:
!pip install -U transformers datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
   

In [ ]:
!pip install -U torch tensorboard
!pip install -U transformers datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
   

In [ ]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/train.jsonl", split="train")

print(dataset)
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 3
})
{'messages': [{'role': 'system', 'content': 'You are a practical SEO and web performance assistant. Give clear priority-based fixes.'}, {'role': 'user', 'content': 'LCP is 5.2s and the hero image is 3.4MB. What should I fix first?'}, {'role': 'assistant', 'content': 'Fix the hero image first because it is likely the main LCP element. Compress it, convert it to WebP or AVIF, preload it, set width and height, and do not lazy-load it if it appears above the fold.'}]}


In [ ]:
for i, sample in enumerate(dataset):
    print(f"\nExample {i+1}")
    for msg in sample["messages"]:
        print(msg["role"], ":", msg["content"][:120])


Example 1
system : You are a practical SEO and web performance assistant. Give clear priority-based fixes.
user : LCP is 5.2s and the hero image is 3.4MB. What should I fix first?
assistant : Fix the hero image first because it is likely the main LCP element. Compress it, convert it to WebP or AVIF, preload it,

Example 2
system : You are a practical SEO and web performance assistant. Give clear priority-based fixes.
user : PageSpeed shows render-blocking CSS and unused JavaScript. What should the developer do?
assistant : Start with render-blocking CSS because it delays first paint. Inline critical CSS, defer non-critical styles, remove unu

Example 3
system : You are a practical SEO and web performance assistant. Give clear priority-based fixes.
user : GSC clicks dropped by 25%, but impressions stayed the same and average position barely changed. What does that mean?
assistant : This looks more like a CTR problem than a ranking problem. Check title tags, meta descriptions, SERP comp

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully:", MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded successfully: google/gemma-4-E2B-it


In [ ]:
import torch
import transformers

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

Torch: 2.10.0+cu128
CUDA: 12.8
Transformers: 5.6.2
GPU available: True


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a practical SEO and web performance assistant. Give clear priority-based fixes."
    },
    {
        "role": "user",
        "content": "LCP is 5.8s and the hero image is 4MB. What should I fix first?"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

system
You are a practical SEO and web performance assistant. Give clear priority-based fixes.
user
LCP is 5.8s and the hero image is 4MB. What should I fix first?
model
This is a classic, high-impact performance bottleneck. **The priority should be to address the Hero Image size immediately.**

Here is the priority-based action plan:

---

### 🥇 Priority 1: Optimize the Hero Image (Highest Impact)

**The Problem:** A 4MB image for a hero element is almost certainly the primary cause of a slow Largest Contentful Paint (LCP). Large images take a long time to download, especially on mobile networks.

**Action Steps (In Order of Preference):**

1. **Compression:** Use a modern image optimization tool (like TinyPNG, Squoosh, or your CMS/hosting platform's built-in tools) to compress the image *without noticeable quality loss*.
    * **Goal:** Reduce the file size drastically (aim for under 300KB, ideally much less).
2. **Modern Format Conversion:** Convert the image to a modern, efficient 